# Fuzzy Logic Model - Habitat Suitability for Plecoptera: Perlidae

This notebook builds a Mamdani fuzzy inference system that predicts the **presence/absence** of *Plecoptera: Perlidae* in the Cali River from five physicochemical predictors (BOD5, DO, turbidity, conductivity and TDS).

Predictor membership functions are defined from expert criteria, and the rule base is generated automatically (one rule per observation). The model is evaluated **in sample** (see the Limitations section).

## Data Loading

In [ ]:
import pandas as pd

# Cali River macroinvertebrate dataset (source: CVC)
file_path = "../../data/DB - Macroinvertebrados.xlsx"
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()
df

In [ ]:
# Keep the predictors and the binary Perlidae response
columns_needed = ['OD', 'DBO5', 'SDT', 'Turbiedad', 'Conductividad', 'Perlidae']
filtered_df = df[columns_needed]

print("First rows of the filtered dataset:")
display(filtered_df.head())
print("\nDescriptive statistics:")
display(filtered_df.describe())

### Outlier removal

Excessively high values are trimmed with an IQR rule using a tolerant upper bound (2.5 x IQR). After this step the dataset contains 14 observations.

In [ ]:
def remove_excessive_outliers(df, column, margin=2.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + margin * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

columns_needed = ['DBO5', 'Turbiedad', 'Conductividad', 'OD', 'SDT', 'Perlidae']
filtered_df = df[columns_needed]
filtered_df_cleaned = filtered_df.copy()
for column in ['DBO5', 'Turbiedad', 'Conductividad', 'OD', 'SDT']:
    filtered_df_cleaned = remove_excessive_outliers(filtered_df_cleaned, column, margin=2.5)

print(f"Original dataset size: {len(filtered_df)}")
print(f"Size after outlier removal: {len(filtered_df_cleaned)}")
print("\nDescriptive statistics of the clean dataset:")
print(filtered_df_cleaned.describe())

## Model Definition

The predictor membership functions are identical to those used in the other fuzzy models. The output is a binary fuzzy variable (Absent / Present) with a 0.5 membership threshold.

In [ ]:
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Input variables (antecedents): universes and triangular membership functions
dbo5 = ctrl.Antecedent(np.arange(0, 9.1, 0.1), 'DBO5')
od = ctrl.Antecedent(np.arange(5, 7.6, 0.1), 'OD')
turbiedad = ctrl.Antecedent(np.arange(0, 31.1, 0.1), 'Turbiedad')
conductividad = ctrl.Antecedent(np.arange(50, 501, 1), 'Conductividad')
sdt = ctrl.Antecedent(np.arange(30, 131, 1), 'SDT')

# Biochemical Oxygen Demand (BOD5)
dbo5['Bajo'] = fuzz.trimf(dbo5.universe, [0, 2, 3])
dbo5['Medio'] = fuzz.trimf(dbo5.universe, [2.5, 4, 6])
dbo5['Alto'] = fuzz.trimf(dbo5.universe, [5, 7, 9])

# Dissolved Oxygen (DO)
od['Bajo'] = fuzz.trimf(od.universe, [4, 5, 5.8])
od['Medio'] = fuzz.trimf(od.universe, [5.5, 6.3, 6.9])
od['Alto'] = fuzz.trimf(od.universe, [6.5, 7, 7.5])

# Turbidity
turbiedad['Baja'] = fuzz.trimf(turbiedad.universe, [0, 2, 5])
turbiedad['Media'] = fuzz.trimf(turbiedad.universe, [4, 8, 12])
turbiedad['Alta'] = fuzz.trimf(turbiedad.universe, [10, 20, 31])

# Conductivity
conductividad['Baja'] = fuzz.trimf(conductividad.universe, [50, 80, 150])
conductividad['Media'] = fuzz.trimf(conductividad.universe, [100, 250, 400])
conductividad['Alta'] = fuzz.trimf(conductividad.universe, [300, 450, 500])

# Total Dissolved Solids (TDS)
sdt['Bajo'] = fuzz.trimf(sdt.universe, [30, 50, 70])
sdt['Medio'] = fuzz.trimf(sdt.universe, [60, 90, 110])
sdt['Alto'] = fuzz.trimf(sdt.universe, [100, 120, 130])

# Output variable (consequent): presence/absence of Plecoptera: Perlidae
perlidae = ctrl.Consequent(np.arange(0, 1.1, 0.1), 'Perlidae')
perlidae['Ausente'] = fuzz.trimf(perlidae.universe, [0, 0, 0.5])
perlidae['Presente'] = fuzz.trimf(perlidae.universe, [0.5, 1, 1])

## Predictor Categorisation

In [ ]:
import skfuzzy as fuzz

# Generic helper: assign a crisp value to the linguistic category with the
# highest membership degree under the previously defined membership functions.
def categorize_variable(value, universe, mf_dict):
    memberships = {cat: fuzz.interp_membership(universe, mf, value)
                   for cat, mf in mf_dict.items()}
    return max(memberships, key=memberships.get)

def categorize_dbo5(value):
    return categorize_variable(value, dbo5.universe,
        {'Bajo': dbo5['Bajo'].mf, 'Medio': dbo5['Medio'].mf, 'Alto': dbo5['Alto'].mf})

def categorize_od(value):
    return categorize_variable(value, od.universe,
        {'Bajo': od['Bajo'].mf, 'Medio': od['Medio'].mf, 'Alto': od['Alto'].mf})

def categorize_turbiedad(value):
    return categorize_variable(value, turbiedad.universe,
        {'Baja': turbiedad['Baja'].mf, 'Media': turbiedad['Media'].mf, 'Alta': turbiedad['Alta'].mf})

def categorize_conductividad(value):
    return categorize_variable(value, conductividad.universe,
        {'Baja': conductividad['Baja'].mf, 'Media': conductividad['Media'].mf, 'Alta': conductividad['Alta'].mf})

def categorize_sdt(value):
    return categorize_variable(value, sdt.universe,
        {'Bajo': sdt['Bajo'].mf, 'Medio': sdt['Medio'].mf, 'Alto': sdt['Alto'].mf})

def categorize_row(row):
    return {
        'DBO5': categorize_dbo5(row['DBO5']),
        'OD': categorize_od(row['OD']),
        'Turbiedad': categorize_turbiedad(row['Turbiedad']),
        'Conductividad': categorize_conductividad(row['Conductividad']),
        'SDT': categorize_sdt(row['SDT']),
        'Perlidae': row['Perlidae'],
    }

categorized_df = filtered_df_cleaned.apply(categorize_row, axis=1, result_type='expand')
print(categorized_df)

## Rule Generation

One fuzzy rule per observation: the antecedent is the conjunction of the categorised predictors and the consequent is Present/Absent according to the observed value.

In [ ]:
rules = []
for _, row in categorized_df.iterrows():
    antecedent = (
        dbo5[row['DBO5']] &
        od[row['OD']] &
        turbiedad[row['Turbiedad']] &
        conductividad[row['Conductividad']] &
        sdt[row['SDT']]
    )
    consequent = perlidae['Presente'] if row['Perlidae'] == 1 else perlidae['Ausente']
    rules.append(ctrl.Rule(antecedent, consequent))

perlidae_ctrl = ctrl.ControlSystem(rules)
perlidae_sim = ctrl.ControlSystemSimulation(perlidae_ctrl)
print(f"{len(rules)} rules were generated for the fuzzy system.")

## Evaluation

The model is run on every observation (in-sample). The defuzzified output is converted to a binary class using a threshold of 0.499.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

predictions = []
real_values = filtered_df_cleaned['Perlidae']

for _, row in filtered_df_cleaned.iterrows():
    try:
        perlidae_sim.input['DBO5'] = row['DBO5']
        perlidae_sim.input['OD'] = row['OD']
        perlidae_sim.input['Turbiedad'] = row['Turbiedad']
        perlidae_sim.input['Conductividad'] = row['Conductividad']
        perlidae_sim.input['SDT'] = row['SDT']
        perlidae_sim.compute()
        prediction = perlidae_sim.output['Perlidae']
        predictions.append(1 if prediction >= 0.499 else 0)
    except Exception as e:
        print(f"Error with data: {row.to_dict()} - {e}")
        predictions.append(0)

print("Confusion matrix:")
print(confusion_matrix(real_values, predictions))
print("\nClassification report:")
print(classification_report(real_values, predictions))
print("\nAccuracy:")
print(accuracy_score(real_values, predictions))

### Performance indicators

In [ ]:
from sklearn.metrics import roc_auc_score, cohen_kappa_score
from sklearn.preprocessing import label_binarize

cm = confusion_matrix(real_values, predictions)
print("Confusion matrix:")
print(cm)

cci = accuracy_score(real_values, predictions)
print(f"Correctly Classified Instances (CCI): {cci:.4f}")

kappa = cohen_kappa_score(real_values, predictions)
print(f"Cohen's Kappa: {kappa:.4f}")

tp = cm[1, 1]; tn = cm[0, 0]; fp = cm[0, 1]; fn = cm[1, 0]
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")

real_bin = label_binarize(real_values, classes=[0, 1])
pred_bin = label_binarize(predictions, classes=[0, 1])
print(f"Area Under the Curve (AUC): {roc_auc_score(real_bin, pred_bin):.4f}")

## Visualisation

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

fpr, tpr, _ = roc_curve(real_bin.ravel(), pred_bin.ravel())
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--', label='Baseline (AUC = 0.50)')
plt.title("ROC Curve - Plecoptera: Perlidae Fuzzy Model")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
import numpy as np

# Real vs predicted presence per observation
real_array = filtered_df_cleaned['Perlidae'].values
indices = np.arange(1, len(real_array) + 1)

plt.figure(figsize=(10, 6))
plt.bar(indices - 0.2, real_array, width=0.4, label="Real", color='blue')
plt.bar(indices + 0.2, predictions, width=0.4, label="Predicted", color='orange')
plt.xlabel("Observations")
plt.ylabel("Presence (0 = Absent, 1 = Present)")
plt.title("Real vs Predicted Presence")
plt.xticks(indices)
plt.legend()
plt.show()

In [ ]:
import seaborn as sns

cm = confusion_matrix(real_values, predictions)
plt.figure(figsize=(3.5, 3))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', annot_kws={"size": 16},
            xticklabels=['Absent', 'Present'], yticklabels=['Absent', 'Present'])
plt.xlabel("Predicted")
plt.ylabel("Real")
plt.title("Confusion Matrix")
plt.show()

## Limitations

Rules were derived from the same observations used for evaluation. Performance metrics therefore reflect in-sample fit and should not be interpreted as evidence of out-of-sample generalisation capacity.

A proper out-of-sample assessment (e.g. leave-one-out cross-validation) is a future improvement, not something implemented here.